Conversione dataframe in tensori

In [ ]:
#esercitazione 4 rete convoluzionale

import torch
from torch import nn
import torchnn as utils
from torch.utils.data import Dataset
import numpy as np

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.X = self.X.unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.y[index]
    
train_dataloader, val_dataloader, test_dataloader = utils.make_dataloaders(
    MyDataset(np.array(X_train), np.array(y_train)),
    MyDataset(np.array(X_val), np.array(y_val)),
    MyDataset(np.array(X_test), np.array(y_test)),
)

#preappello rete normale

import torch
from torch import nn
from torch.utils.data import Dataset, TensorDataset
import torchnn as utils

target_scaler = StandardScaler()
y_train = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test = target_scaler.transform(y_test.values.reshape(-1, 1))
y_val = target_scaler.transform(y_val.values.reshape(-1, 1))

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(np.array(X), dtype=torch.float32)
        self.y = torch.tensor(np.array(y), dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.y[index]
        
train_dataloader, val_dataloader, test_dataloader = utils.make_dataloaders(
    MyDataset(X_train, y_train), MyDataset(X_val, y_val), MyDataset(X_test, y_test)
)

#rete neurale 12_06_2024

import torch 
from torch.utils.data import Dataset
import torchnn as utils

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        print(self.X.shape)
        self.y = torch.tensor(y, dtype=torch.long)
        print(self.y.shape)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.y[index]
    
train_dataloader, val_dataloader, test_dataloader = utils.make_dataloaders(
    MyDataset(np.array(X_train), np.array(y_train)),
    MyDataset(np.array(X_val), np.array(y_val)),
    MyDataset(np.array(X_test), np.array(y_test)),
)



creazione rete neurale

In [1]:
#esercitazione 4 rete convulozionale

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv1d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AvgPool1d(2),
            nn.Flatten(),
            
            nn.Linear(64 * (X_train.shape[1] // 2), 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, len(label_encoder.classes_)),
            nn.LogSoftmax(dim=1)
        )
        
    def forward(self, x):
        return self.layers(x)
    


#pre appello rete neuralee

class Net(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        w = 64
        
        self.layers = nn.Sequential(
            nn.Linear(X_train.shape[1], w),
            nn.ReLU(),
            
            nn.Dropout(0.2),
            nn.Linear(w, w),
            nn.ReLU(),
            
            nn.Dropout(0.2),
            nn.Linear(w, w),
            nn.ReLU(),
            
            nn.Linear(w, 1)
        )
        
    def forward(self, x):
        return self.layers(x)
    
model = Net()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=30)
early_stopping = utils.EarlyStopping(patience=10, min_delta=0.001)


#rete neurale 12_06_2024

from torch import nn
class Net(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        w = 128
        self.layers = nn.Sequential(
            nn.Linear(X_train.shape[1], w),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(w,w),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(w,w),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(w, 2),
            nn.LogSoftmax(dim=1)      
        )
        
    def forward(self, x):
        return self.layers(x)
    
model = Net()
device = 'cpu'
model = model.to(device)

optimizer = torch.optim.RMSprop(model.parameters(), centered=True, weight_decay=1e-4, momentum=0.1)
early_stopper = utils.EarlyStopping(patience=5, min_delta=0.01)
print(model.parameters())

NameError: name 'nn' is not defined

non lo so1

In [ ]:
######################
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

def eval_loop(model, dataloader, device, loss_fn):
    model.eval()

    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, accuracy = 0.0, 0

    y_true = []
    y_pred = []
    y_pred_proba = []
    # context manager che disabilita esplicitamente il calcolo dei gradienti in fase di test
    with torch.no_grad():
        for X, y in dataloader:

            # Spostiamo esplicitamente i tensori sul device di computazione
            X, y = X.to(device), y.to(device)

            pred = model(X)
            test_loss += loss_fn(pred, y).item()

            # L'accuracy sul batch si calcola creando il tensore di dimensione pari al batch
            # per cui i massimi argomenti di ogni predizione sono uguali alla classe predetta
            # per il singolo campione, poi convertendo questo tensore di booleani in un vettore
            # binario di tipo float, sommando i valori 1 ed estraendo lo scalare contenuto nel tensore risultato
            accuracy += (pred.argmax(1) == y).type(torch.float).sum().item()

            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.argmax(1).cpu().numpy())
            y_pred_proba.extend(torch.exp(pred).cpu().numpy())

    test_loss /= num_batches
    accuracy /= size            # Per quanto detto prima, l'accuracy media va calcolata sulla dimensione del data set

    epoch_metrics = {}

    y_pred = np.array(y_pred)
    y_true = np.array(y_true)
    y_pred_proba = np.array(y_pred_proba)
    
    epoch_metrics['f1_score'] = f1_score(y_true, y_pred, average='weighted')
    epoch_metrics['precision'] = precision_score(y_true, y_pred, average='weighted')
    epoch_metrics['recall'] = recall_score(y_true, y_pred, average='weighted')
    epoch_metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='weighted')

    return test_loss, accuracy, epoch_metrics, (y_true, y_pred, y_pred_proba)

#########################################
def eval_loop(model, dataloader, device, loss_fn):
    model.eval()

    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, accuracy = 0.0, 0

    y_true = []
    y_pred = []
    with torch.no_grad():
        for X, y in dataloader:

            # Spostiamo esplicitamente i tensori sul device di computazione
            X, y = X.to(device), y.to(device)

            pred = model(X)
            test_loss += loss_fn(pred, y).item()

            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    test_loss /= num_batches

    # Calcoliamo il dizionario delle metriche per epoca
    epoch_metrics = {}
    epoch_metrics['mse'] = mean_squared_error(y_true, y_pred)
    epoch_metrics['r2'] = r2_score(y_true, y_pred)
    return test_loss, epoch_metrics, y_true, y_pred

########################################################

from sklearn.metrics import roc_auc_score

def eval_loop(model, 
              dataloader, 
              device, 
              loss_fn):
    model.eval()

    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, accuracy = 0.0, 0

    y_true = []
    y_pred = []
    y_pred_probs = []
    # context manager che disabilita esplicitamente il calcolo dei gradienti in fase di test
    with torch.no_grad():
        for X, y in dataloader:

            # Spostiamo esplicitamente i tensori sul device di computazione
            X, y = X.to(device), y.to(device)

            pred = model(X)
            test_loss += loss_fn(pred, y).item()

            # L'accuracy sul batch si calcola creando il tensore di dimensione pari al batch
            # per cui i massimi argomenti di ogni predizione sono uguali alla classe predetta
            # per il singolo campione, poi convertendo questo tensore di booleani in un vettore
            # binario di tipo float, sommando i valori 1 ed estraendo lo scalare contenuto nel tensore risultato
            accuracy += (pred.argmax(1) == y).type(torch.float).sum().item()

            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.argmax(1).cpu().numpy())
            y_pred_probs.extend(torch.exp(pred).cpu().numpy())

    test_loss /= num_batches
    accuracy /= size            # Per quanto detto prima, l'accuracy media va calcolata sulla dimensione del data set
    auc = roc_auc_score(y_true, np.array(y_pred_probs)[:,1])

    return test_loss, accuracy, auc, (y_true, y_pred, y_pred_probs)


non lo so 2

In [ ]:
################################

from tqdm import trange
epochs = 50
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.NLLLoss()
save_best_model = SaveBestModel()
early_stopper = utils.EarlyStopping(patience=5, min_delta=0.01)
train_loss = []
val_loss = []
test_loss = []
accuracy = []

train_batches = len(train_dataloader.batch_sampler)
for epoch in range(1, epochs+1):
    pbar = trange(train_batches)
    pbar.set_description(desc='Epoch {:4d}'.format(epoch))
    
    epoch_train_loss = utils.train_loop(model, train_dataloader, optimizer, device, pbar, loss_fn=criterion)
    train_loss.append(epoch_train_loss)
    
    epoch_val_loss, val_acc, val_metrics, _ = eval_loop(model, val_dataloader, device, criterion)
    val_loss.append(epoch_val_loss)
    accuracy.append(val_acc)
    
    epoch_test_loss, test_acc, test_metrics, _ = eval_loop(model, test_dataloader, device, criterion)
    test_loss.append(epoch_test_loss)
    
    print(f"Epoch {epoch}: train_loss={epoch_train_loss:.4f}, val_loss={epoch_val_loss:.4f}, val_acc={val_acc:.4f}, test_loss={epoch_test_loss:.4f}\nval_accuracy={val_acc:.4f}")
    
    save_best_model(epoch_val_loss, model, optimizer, epoch_train_loss, epoch_val_loss, epoch_test_loss, val_acc)
    early_stopper(epoch_val_loss)
    
    if early_stopper.early_stop:
        print("Early stopping")
        break
#############################################


from tqdm import trange
epochs = 30
train_batches = len(train_dataloader.batch_sampler)
train_loss = []
metrics = {
    'mse': [],
    'r2': []
}
save_best_model = SaveBestModel()
for epoch in range(1, epochs+1):
    pbar = trange(train_batches)
    pbar.set_description(desc='Epoch {:4d}'.format(epoch))
    
    epoch_train_loss = utils.train_loop(model, train_dataloader, optimizer, device, pbar, loss_fn=criterion)
    train_loss.append(epoch_train_loss)
    
    val_loss, epoch_metrics, _, _ = eval_loop(model, val_dataloader, device, loss_fn=criterion)
    print(f"Epoch {epoch} - Train Loss: {epoch_train_loss:.4f} - Val Loss: {val_loss:.4f}")
    print(f"MSE: {epoch_metrics['mse']:.4f} - R2: {epoch_metrics['r2']:.4f}")
    
    metrics['mse'].append(epoch_metrics['mse'])
    metrics['r2'].append(epoch_metrics['r2'])
    
    save_best_model(model, optimizer, epoch, epoch_train_loss, val_loss, metrics)
    
    early_stopping(val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break
    
    if scheduler is not None:
        scheduler.step()
    
    ##################################################
    
    from tqdm import trange
epochs = 30
loss_fn = nn.NLLLoss()

train_loss, validation_loss, test_loss = [], [], []
accuracy = []
metrics = {'auc': []}

train_batches = len(train_dataloader.batch_sampler)
saver = SaveBestModel()
for epoch in range(1, epochs + 1):
    pbar = trange(train_batches)
    pbar.set_description(desc='Epoch {:4d}'.format(epoch))
    
    epoch_train_loss = utils.train_loop(model, train_dataloader, optimizer, device, pbar, loss_fn=loss_fn)
    train_loss.append(epoch_train_loss)
    
    epoch_validate_loss, epoch_accuracy, auc, _ = eval_loop(model, val_dataloader, device, loss_fn=loss_fn)
    validation_loss.append(epoch_validate_loss)
    metrics['auc'].append(auc)
    
    print(f"Train loss: {epoch_train_loss} | Val loss: {epoch_validate_loss}")
    print(f"Auc: {auc} | accuracy: {epoch_accuracy}")
    saver(epoch_accuracy, model, optimizer, epoch, train_loss, validation_loss, test_loss, metrics=metrics)
    
    early_stopper(validation_loss=epoch_validate_loss)
    if early_stopper.early_stop:
        break

saver.save('./prova2024.pth')

salvataggio modello migliore

In [ ]:
#####################################

import copy
        
class SaveBestModel():
    def __init__(self):
        self.best_model_state = None
        self.best_optimizer_state = None
        self.best_loss = float('inf')
        
    def __call__(self, epoch_loss, model, optimizer, train_loss, val_loss, test_loss, accuracy):
        if epoch_loss < self.best_loss:
            self.best_loss = epoch_loss
            self.best_model_state = copy.deepcopy(model.state_dict())
            self.best_optimizer_state = copy.deepcopy(optimizer.state_dict())
            self.train_loss = train_loss
            self.val_loss = val_loss
            self.test_loss = test_loss
            self.accuracy = accuracy
            
    def save(self, path):
        torch.save({
            "model_state_dict": self.best_model_state,
            "optimizer_state_dict": self.best_optimizer_state,
            "train_loss": self.train_loss,
            "val_loss": self.val_loss,
            "test_loss": self.test_loss,
            "accuracy": self.accuracy
        },path)
######################################################################

import copy
class SaveBestModel:
    def __init__(self):
        self.best_loss = float('inf')
        self.best_model_state = None
        self.best_optimizer_state = None        
    def __call__(self, model, optimizer, epoch, train_loss, val_loss, metrics):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.model_state = copy.deepcopy(model.state_dict())
            self.optimizer_state = copy.deepcopy(optimizer.state_dict())
            self.epoch = epoch
            self.train_loss = train_loss
            self.val_loss = val_loss
            self.metrics = metrics
            
    def save(self, path):
        torch.save({
            'model_state_dict': self.model_state,
            'optimizer_state_dict': self.optimizer_state,
            'epoch': self.epoch,
            'train_loss': self.train_loss,
            'val_loss': self.val_loss,
            'metrics': self.metrics
        }, path)
############################################################

import copy
class SaveBestModel():
    def __init__(self):
        self.model_state = None
        self.optimizer_state = None
        self.best_accuracy = -1
    
    def __call__(self, accuracy, model, optimizer, epoch, train_loss, val_loss, test_loss, metrics):
        if accuracy > self.best_accuracy:
            self.best_accuracy = accuracy
            self.model_state = copy.deepcopy(model.state_dict())
            self.optimizer_state = copy.deepcopy(optimizer.state_dict())
            self.epoch = epoch
            self.train_loss = train_loss
            self.val_loss = val_loss
            self.test_loss = test_loss
            self.metrics = metrics
            
    def save(self, path):
        torch.save({
            'model_state_dict': self.model_state,
            'optimizer_state_dict': self.optimizer_state,
            'epoch': self.epoch,
            'train_loss': self.train_loss,
            'val_loss': self.val_loss,
            'test_loss': self.test_loss,
            'metrics': self.metrics
        }, path)